In [0]:
from pyspark.sql import functions as F

CATALOG_NAME = "workspace"
SCHEMA_NAME = "metadata"

METADATA_PROFILE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.metadata_profile"
DQ_RULES_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.dq_rule_recommendations"

In [0]:
metadata_profile_df = spark.table(METADATA_PROFILE_TABLE)

display(metadata_profile_df)

In [0]:
required_columns = [
    "table_name",
    "col_name",
    "data_type",
    "semantic_type",
    "null_percent",
    "distinct_count",
    "total_rows",
    "governance_tag"
]

missing_columns = []

for col in required_columns:
    if col not in metadata_profile_df.columns:
        missing_columns.append(col)

if missing_columns:
    raise ValueError(
        f"Missing required columns in metadata profile: {missing_columns}"
    )

print("Metadata profile loaded successfully.")

In [0]:
display(
    metadata_profile_df
    .select(
        "table_name",
        "col_name",
        "data_type",
        "semantic_type",
        "null_percent",
        "distinct_count",
        "total_rows",
        "governance_tag"
    )
    .orderBy("table_name", "col_name")
)

In [0]:
metadata_rows = [
    row.asDict()
    for row in metadata_profile_df.collect()
]

print(f"Loaded {len(metadata_rows)} metadata rows for rule recommendation.")

In [0]:
# rule skeleton function

def create_rule(table_name, col_name, rule_type, rule_description, rule_sql, severity):
    return {
        "table_name": table_name,
        "col_name": col_name,
        "rule_type": rule_type,
        "rule_description": rule_description,
        "rule_sql": rule_sql,
        "severity": severity
    }

In [0]:
# recommend data quality rules for each column
# based on semantic type, column name, null percentage, distinct count, and total row count


def recommend_column_rules(profile_row):
    table_name = profile_row["table_name"]
    col_name = profile_row["col_name"]
    semantic_type = profile_row["semantic_type"]
    total_rows = profile_row["total_rows"]
    distinct_count = profile_row["distinct_count"]

    rules = []

    # should not be null
    important_semantic_types = [
        "identifier",
        "email",
        "phone_number",
        "date_of_birth",
        "date",
        "currency",
        "numeric_measure",
        "age",
        "gender"
    ]

    # excluded from null checks
    optional_not_null_exclusions = [
        "delivery_date"
    ]

    if semantic_type in important_semantic_types and col_name not in optional_not_null_exclusions:
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="not_null",
                rule_description=f"{table_name}.{col_name} should not be null or blank.",
                rule_sql=f"{col_name} IS NOT NULL AND CAST({col_name} AS STRING) <> ''",
                severity="high"
            )
        )

    # high cardinality identifier columns are likely primary keys
    if semantic_type == "identifier" and total_rows > 0:
        uniqueness_ratio = distinct_count / total_rows

        if uniqueness_ratio > 0.90:
            rules.append(
                create_rule(
                    table_name=table_name,
                    col_name=col_name,
                    rule_type="unique",
                    rule_description=f"{table_name}.{col_name} appears to be a primary key candidate and should be unique.",
                    rule_sql=f"COUNT({col_name}) = COUNT(DISTINCT {col_name})",
                    severity="high"
                )
            )

    # email format rule.
    if semantic_type == "email":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="regex_format",
                rule_description=f"{table_name}.{col_name} should match a valid email format.",
                rule_sql=f"{col_name} RLIKE '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\\\.[A-Za-z]{{2,}}$'",
                severity="high"
            )
        )

    # phone format rule
    if semantic_type == "phone_number":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="regex_format",
                rule_description=f"{table_name}.{col_name} should match the expected phone format.",
                rule_sql=f"{col_name} RLIKE '^[0-9]{{3}}-[0-9]{{3}}-[0-9]{{4}}$'",
                severity="medium"
            )
        )

    # zip code format rule
    if semantic_type == "zip_code":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="regex_format",
                rule_description=f"{table_name}.{col_name} should be a valid 5-digit ZIP code.",
                rule_sql=f"{col_name} RLIKE '^[0-9]{{5}}$'",
                severity="medium"
            )
        )

    # age range rule
    if col_name == "age":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="valid_range",
                rule_description=f"{table_name}.{col_name} should be between 0 and 120.",
                rule_sql=f"{col_name} BETWEEN 0 AND 120",
                severity="medium"
            )
        )

    # currency fields should not be negative
    if semantic_type == "currency":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="valid_range",
                rule_description=f"{table_name}.{col_name} should be greater than or equal to 0.",
                rule_sql=f"{col_name} >= 0",
                severity="medium"
            )
        )

    # stock quantity should not be negative
    if col_name == "stock_quantity":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="valid_range",
                rule_description=f"{table_name}.{col_name} should be greater than or equal to 0.",
                rule_sql=f"{col_name} >= 0",
                severity="medium"
            )
        )

    # date of birth should not be in the future
    if semantic_type == "date_of_birth":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="date_not_future",
                rule_description=f"{table_name}.{col_name} should not be in the future.",
                rule_sql=f"try_cast({col_name} AS DATE) <= CURRENT_DATE()",
                severity="medium"
            )
        )

    return rules

In [0]:
# rules for known values

def recommend_known_value_rules(profile_row):
    table_name = profile_row["table_name"]
    col_name = profile_row["col_name"]

    rules = []

    # gender needs to be from list
    if col_name == "gender":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="valid_values",
                rule_description=f"{table_name}.{col_name} should contain only accepted gender values.",
                rule_sql=(
                    f"{col_name} IN "
                    "('Male', 'Female', 'Non-Binary', 'Prefer not to say')"
                ),
                severity="low"
            )
        )

    # order status needs to be in list
    if col_name == "order_status":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="valid_values",
                rule_description=f"{table_name}.{col_name} should contain only accepted order statuses.",
                rule_sql=(
                    f"{col_name} IN "
                    "('placed', 'processing', 'shipped', 'delivered', 'cancelled', 'returned')"
                ),
                severity="medium"
            )
        )

    # valid payment method
    if col_name == "payment_method":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="valid_values",
                rule_description=f"{table_name}.{col_name} should contain only accepted payment methods.",
                rule_sql=(
                    f"{col_name} IN "
                    "('credit_card', 'debit_card', 'paypal', 'apple_pay', 'gift_card')"
                ),
                severity="medium"
            )
        )

    # country must be US
    if col_name == "country":
        rules.append(
            create_rule(
                table_name=table_name,
                col_name=col_name,
                rule_type="valid_values",
                rule_description=f"{table_name}.{col_name} should contain only accepted country values.",
                rule_sql=f"{col_name} IN ('USA')",
                severity="low"
            )
        )

    return rules

In [0]:
# table level rules
# not yet implemented


# customer id in orders and customers should match

# ship date must be after order date

# delivery date must be after ship date

In [0]:
# collect all rule recommendations into Spark DataFrame

metadata_rows = [
    row.asDict()
    for row in metadata_profile_df.collect()
]

rule_rows = []

for row in metadata_rows:
    rule_rows.extend(
        recommend_column_rules(row)
    )

    rule_rows.extend(
        recommend_known_value_rules(row)
    )

# rule_rows.extend(
#     recommend_table_level_rules()
# )

dq_rules_df = spark.createDataFrame(rule_rows)

display(dq_rules_df)

In [0]:
dq_rules_df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.dq_rule_recommendations")

In [0]:
display(
    metadata_profile_df
    .groupBy("table_name", "governance_tag")
    .count()
    .orderBy("table_name", "governance_tag")
)